## 1. Veri Yükleme

In [8]:
import numpy as np
import os

PROCESSED_DIR = '../datasets/processed'

indian_img = np.load(os.path.join(PROCESSED_DIR, 'indian_pines_img.npy'))
indian_gt  = np.load(os.path.join(PROCESSED_DIR, 'indian_pines_gt.npy'))
pavia_img  = np.load(os.path.join(PROCESSED_DIR, 'pavia_university_img.npy'))
pavia_gt   = np.load(os.path.join(PROCESSED_DIR, 'pavia_university_gt.npy'))
salinas_img = np.load(os.path.join(PROCESSED_DIR, 'salinas_img.npy'))
salinas_gt  = np.load(os.path.join(PROCESSED_DIR, 'salinas_gt.npy'))

print(indian_img.shape, pavia_img.shape, salinas_img.shape)


(145, 145, 200) (610, 340, 103) (512, 217, 204)


## 2. Global Normalizasyon Adımı

Bant bazlı normalizasyon yaparsak gürültülü bantlarda (0.001 -> 0, 0.002 -> 1) varyans çok yüksek çıkar. PCA bunu önemli bir sinyal zannedip ilk temel bileşenlere yerleştirebilir.

Global normalizasyonda ise bantların birbirine göre olan doğal varyans oranlarını korumuş oluruz.

In [9]:
def normalize(img):
    h, w, b = img.shape
    flat = img.reshape(-1, b).astype(np.float32)
    global_min = flat.min()
    global_max = flat.max()
    return ((flat - global_min) / (global_max - global_min)).reshape(h, w, b)

indian_norm  = normalize(indian_img)
pavia_norm   = normalize(pavia_img)
salinas_norm = normalize(salinas_img)

np.save(os.path.join(PROCESSED_DIR, 'indian_pines_norm.npy'), indian_norm)
np.save(os.path.join(PROCESSED_DIR, 'pavia_university_norm.npy'), pavia_norm)
np.save(os.path.join(PROCESSED_DIR, 'salinas_norm.npy'), salinas_norm)

print(f"Indian: min={indian_norm.min():.2f} max={indian_norm.max():.2f}")
print(f"Pavia:  min={pavia_norm.min():.2f} max={pavia_norm.max():.2f}")
print(f"Salinas: min={salinas_norm.min():.2f} max={salinas_norm.max():.2f}")


Indian: min=0.00 max=1.00
Pavia:  min=0.00 max=1.00
Salinas: min=0.00 max=1.00


## 3. Gaussian Filtreleme Yöntemi

Filtre sadece 'h' ve 'w' eksenlerine uygulanmalı yani uzamsal olmalı, sigma değeri olarak (0.5, 1.0, 1.5) 3 değeri alıyoruz ileride test edeceğiz. Reflect padding yöntemi de köşedeki değerler için uygulanıyor.

In [10]:
from scipy.ndimage import gaussian_filter

def gaussian_filter_spatial(img, sigma):
    # sigma=(sigma, sigma, 0) → sadece h,w eksenine uygula, bant eksenine dokunma
    return gaussian_filter(img, sigma=(sigma, sigma, 0), mode='reflect')

for dataset_name, img, prefix in [
    ('Indian Pines', indian_norm, 'indian_pines'),
    ('Pavia University', pavia_norm, 'pavia_university'),
    ('Salinas', salinas_norm, 'salinas'),
]:
    for sigma in [0.5, 1.0, 1.5]:
        filtered = gaussian_filter_spatial(img, sigma)
        np.save(os.path.join(PROCESSED_DIR, f'{prefix}_gauss{sigma}.npy'), filtered)
        print(f"✓ {dataset_name} sigma={sigma} kaydedildi")


✓ Indian Pines sigma=0.5 kaydedildi
✓ Indian Pines sigma=1.0 kaydedildi
✓ Indian Pines sigma=1.5 kaydedildi
✓ Pavia University sigma=0.5 kaydedildi
✓ Pavia University sigma=1.0 kaydedildi
✓ Pavia University sigma=1.5 kaydedildi
✓ Salinas sigma=0.5 kaydedildi
✓ Salinas sigma=1.0 kaydedildi
✓ Salinas sigma=1.5 kaydedildi


## 4.Varyans Temelli ve Sabit Bileşenli PCA Uygulama

Varyans Temelli PCA ile %99.5 veriyi korumak istediğimizde gaussian filtresinde kullandığımız her sigma değeri için farklı bir bant sayısına indirgenme gerçekleşiyor. 

Indian Pines (0.5 -> 26 bant, 1.0 -> 15 bant, 1.5 -> 13 bant)

Pavia University (0.5 -> 5 bant, 1.0 -> 4 bant, 1.5 -> 4 bant)

Salinas (0.5 -> 4 bant, 1.0 -> 4 bant, 1.5 -> 4 bant)

Fakat özellikle Salinas ve Pavia University'deki 4, 5 bantlar CNN ve ViT modelleri için az sayıda olduğu için Sabit Bileşenli PCA ile 30 bant kullanılması uygun görüldü.

In [ ]:
"""from sklearn.decomposition import PCA

for dataset_name, prefix in [
    ('Indian Pines', 'indian_pines'),
    ('Pavia University', 'pavia_university'),
    ('Salinas', 'salinas'),
]:
    for sigma in [0.5, 1.0, 1.5]:
        img = np.load(os.path.join(PROCESSED_DIR, f'{prefix}_gauss{sigma}.npy'))
        h, w, b = img.shape
        flat = img.reshape(-1, b)

        pca = PCA(n_components=0.995)
        flat_pca = pca.fit_transform(flat)
        n_components = flat_pca.shape[1]

        img_pca = flat_pca.reshape(h, w, n_components)
        out_name = f'{prefix}_gauss{sigma}_pca{n_components}.npy'
        np.save(os.path.join(PROCESSED_DIR, out_name), img_pca)
        print(f"✓ {dataset_name} sigma={sigma} → {b} bant → {n_components} bant | {out_name}")"""


✓ Indian Pines sigma=0.5 → 200 bant → 26 bant | indian_pines_gauss0.5_pca26.npy
✓ Indian Pines sigma=1.0 → 200 bant → 15 bant | indian_pines_gauss1.0_pca15.npy
✓ Indian Pines sigma=1.5 → 200 bant → 13 bant | indian_pines_gauss1.5_pca13.npy
✓ Pavia University sigma=0.5 → 103 bant → 5 bant | pavia_university_gauss0.5_pca5.npy
✓ Pavia University sigma=1.0 → 103 bant → 4 bant | pavia_university_gauss1.0_pca4.npy
✓ Pavia University sigma=1.5 → 103 bant → 4 bant | pavia_university_gauss1.5_pca4.npy
✓ Salinas sigma=0.5 → 204 bant → 4 bant | salinas_gauss0.5_pca4.npy
✓ Salinas sigma=1.0 → 204 bant → 4 bant | salinas_gauss1.0_pca4.npy
✓ Salinas sigma=1.5 → 204 bant → 4 bant | salinas_gauss1.5_pca4.npy


In [12]:
from sklearn.decomposition import PCA

for prefix in ['indian_pines', 'pavia_university', 'salinas']:
    for sigma in [0.5, 1.0, 1.5]:
        img = np.load(os.path.join(PROCESSED_DIR, f'{prefix}_gauss{sigma}.npy'))
        h, w, b = img.shape
        flat = img.reshape(-1, b)
        pca = PCA(n_components=30)
        reduced = pca.fit_transform(flat)
        result = reduced.reshape(h, w, 30)
        np.save(os.path.join(PROCESSED_DIR, f'{prefix}_gauss{sigma}_pca30.npy'), result)
        print(f"✓ {prefix} sigma={sigma} → {result.shape}")


✓ indian_pines sigma=0.5 → (145, 145, 30)
✓ indian_pines sigma=1.0 → (145, 145, 30)
✓ indian_pines sigma=1.5 → (145, 145, 30)
✓ pavia_university sigma=0.5 → (610, 340, 30)
✓ pavia_university sigma=1.0 → (610, 340, 30)
✓ pavia_university sigma=1.5 → (610, 340, 30)
✓ salinas sigma=0.5 → (512, 217, 30)
✓ salinas sigma=1.0 → (512, 217, 30)
✓ salinas sigma=1.5 → (512, 217, 30)
